# 01 · Auditoria e preparação dos dados (Etapa 1)

Gera `dataset/processed/tracks_clean.csv` (e `.parquet`, se o `pyarrow` estiver instalado) a partir do `dataset/spotify-2023.csv`, com relatório do que foi corrigido, excluído ou mantido.

**Regras:** o CSV original nunca é alterado (o hash é conferido no fim). Nenhuma linha é apagada sem registro no log. Rode com *Kernel → Restart & Run All*.

## Configuração e utilitários (Etapa 0)

In [13]:
import hashlib, json, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# --- Caminhos (funciona rodando de notebooks/ ou da raiz do repositório) ---
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV = ROOT / "dataset" / "spotify-2023.csv"
PROCESSED_DIR = ROOT / "dataset" / "processed"
REPORTS_DIR = ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
for d in (PROCESSED_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
ENCODING = "cp1252"

# --- Colunas do CSV original (24) ---
EXPECTED_COLUMNS = [
    "track_name", "artist(s)_name", "artist_count",
    "released_year", "released_month", "released_day",
    "in_spotify_playlists", "in_spotify_charts", "streams",
    "in_apple_playlists", "in_apple_charts",
    "in_deezer_playlists", "in_deezer_charts", "in_shazam_charts",
    "bpm", "key", "mode",
    "danceability_%", "valence_%", "energy_%", "acousticness_%",
    "instrumentalness_%", "liveness_%", "speechiness_%",
]
COUNT_COLS = ["in_spotify_playlists", "in_spotify_charts", "streams",
              "in_apple_playlists", "in_apple_charts",
              "in_deezer_playlists", "in_deezer_charts", "in_shazam_charts"]
PCT_COLS = ["danceability_%", "valence_%", "energy_%", "acousticness_%",
            "instrumentalness_%", "liveness_%", "speechiness_%"]
DATE_COLS = ["released_year", "released_month", "released_day"]
NUMERIC_COLS = COUNT_COLS + PCT_COLS + DATE_COLS + ["artist_count", "bpm"]

# --- Parâmetros de limpeza ---
# A confirmar na página do Kaggle (pendência do data.md). Usada só em age_years.
# Confirmado -> data correta: 26/08/2023
DATA_REFERENCE_DATE = pd.Timestamp("2023-08-26")
PERIOD_BINS = [-float("inf"), 1999, 2017, 2021, 2022, float("inf")]
PERIOD_LABELS = ["pre-2000", "2000-2017", "2018-2021", "2022", "2023"]
# Duplicatas NÃO exatas: "flag" (só marca) | "keep_latest" | "mean" | "drop_all"
NON_EXACT_DUP_POLICY = "flag"
NEAR_DUP_STREAMS_RTOL = 0.05   # diferença relativa de streams para "quase igual"

# --- Conjuntos de variáveis (usados na Etapa 3) ---
FEATURES_A = ["bpm", "key", "mode"] + PCT_COLS
FEATURES_B = FEATURES_A + ["release_period", "artist_count"]
FEATURES_C = FEATURES_B + ["main_artist"]   # só depois de fechar A e B

# --- Log de decisões e tabela de anomalias ---
LOG, ANOMALIES = [], []

def decide(etapa, item, n, acao, motivo=""):
    LOG.append(dict(etapa=etapa, item=item, n=int(n), acao=acao, motivo=motivo))

def add_anomalies(df, mask, column, issue, raw=None):
    for idx in df.index[mask]:
        val = raw[idx] if raw is not None else df.at[idx, column]
        ANOMALIES.append(dict(row_id=int(df.at[idx, "row_id"]), csv_line=int(df.at[idx, "csv_line"]),
                              track_name=df.at[idx, "track_name"], column=column,
                              issue=issue, raw_value=str(val)[:80]))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not RAW_CSV.exists():
    raise FileNotFoundError(f"CSV original não encontrado em {RAW_CSV}")
HASH_BEFORE = sha256_file(RAW_CSV)
print("CSV:", RAW_CSV, "\nsha256:", HASH_BEFORE)

CSV: /home/zana/projects/facul/intML/ecp-records-project/dataset/spotify-2023.csv 
sha256: 26fd1292465877c24a9f5bfc6bf933c0715356a9c7d3197f2c4bc8b1a668a215


## 1. Leitura
`encoding="cp1252"` explícito e tudo lido como texto, para não perder nada na conversão. Células vazias viram `NA`.
`row_id` é a posição (0-based) e `csv_line` é a linha no arquivo (cabeçalho = 1), útil para localizar casos como a linha 574.

In [14]:
raw = pd.read_csv(RAW_CSV, encoding=ENCODING, dtype=str, keep_default_na=False)
raw.columns = [c.strip() for c in raw.columns]
missing_cols = [c for c in EXPECTED_COLUMNS if c not in raw.columns]
assert not missing_cols, f"Colunas esperadas ausentes: {missing_cols}"

raw = raw[EXPECTED_COLUMNS].apply(lambda s: s.str.strip())
raw = raw.where(raw != "", pd.NA)
raw.insert(0, "row_id", np.arange(len(raw)))
raw.insert(1, "csv_line", np.arange(len(raw)) + 2)
N_RAW = len(raw)
print(f"{N_RAW} linhas x {len(EXPECTED_COLUMNS)} colunas originais")
display(raw.head(3))

953 linhas x 24 colunas originais


,row_id,csv_line,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,0,2,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,...,125,B,Major,80,89,83,31,0,8,4
1,1,3,LALA,Myke Towers,1,2023,3,23,1474,48,...,92,C#,Major,71,61,74,7,0,10,4
2,2,4,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,...,138,F,Major,51,32,53,17,0,31,6


### 1.1 Mojibake nos nomes
Se o arquivo foi lido na codificação certa, não deve haver sequências como `Ã©` ou `â€™`.

In [15]:
df = raw.copy()
MOJIBAKE = r"Ã.|â€|Â"
names = df["track_name"].fillna("")
arts = df["artist(s)_name"].fillna("")
moj = names.str.contains(MOJIBAKE, regex=True) | arts.str.contains(MOJIBAKE, regex=True)
add_anomalies(df, moj, "track_name", "possível mojibake")
decide("leitura", "nomes com possível mojibake", moj.sum(), "mantidos e registrados",
       "lido em cp1252; conferir manualmente se houver casos")
print("Nomes com possível mojibake:", int(moj.sum()))
display(df.loc[moj, ["csv_line", "track_name", "artist(s)_name"]])

Nomes com possível mojibake: 0


,csv_line,track_name,artist(s)_name


## 2. Normalização numérica
- Remove o separador de milhares (`,`) das colunas numéricas.
- Converte com `pd.to_numeric(errors="coerce")`.
- Todo valor que existia e virou `NaN` entra na tabela de anomalias. É assim que a linha com `streams` corrompido aparece.

In [16]:
missing_before = {c: df[c].isna() for c in NUMERIC_COLS}
for c in NUMERIC_COLS:
    s = df[c]
    sep = s.str.fullmatch(r"\d{1,3}(,\d{3})+", na=False)
    if sep.any():
        decide("numérico", f"separador de milhares em {c}", sep.sum(), "vírgula removida")
    df[c] = pd.to_numeric(s.str.replace(",", "", regex=False), errors="coerce")
    bad = s.notna() & df[c].isna()
    if bad.any():
        add_anomalies(df, bad, c, "valor não numérico (virou NaN)", raw=s)
        decide("numérico", f"valor não numérico em {c}", bad.sum(), "convertido para NaN e registrado")

df["streams_invalid"] = raw["streams"].notna() & df["streams"].isna()
print("Linhas com streams inválido:", int(df["streams_invalid"].sum()))
display(raw.loc[df["streams_invalid"], ["csv_line", "track_name", "artist(s)_name", "streams"]])
display(pd.DataFrame([x for x in LOG if x["etapa"] == "numérico"]))

Linhas com streams inválido: 1


,csv_line,track_name,artist(s)_name,streams
574,576,Love Grows (Where My Rosemary Goes),Edison Lighthouse,BPM110KeyAModeMajorDanceability53Valence75Ener...


,etapa,item,n,acao,motivo
0,numérico,valor não numérico em streams,1,convertido para NaN e registrado,
1,numérico,separador de milhares em in_deezer_playlists,79,vírgula removida,
2,numérico,separador de milhares em in_shazam_charts,7,vírgula removida,


## 3. Valores ausentes
- `key`: categoria explícita `"Unknown"` + indicador `key_missing`. **Nunca imputar `C`.**
- `in_shazam_charts`: mantém `NaN` + indicador. Essas colunas ficam fora dos modelos principais, então a decisão só pesa na EDA.

In [17]:
for c in EXPECTED_COLUMNS:
    n = int(raw[c].isna().sum())
    if n:
        decide("ausentes", f"{c} vazio", n,
               "tratado abaixo" if c in ("key", "in_shazam_charts") else "mantido como NaN")

df["key_missing"] = df["key"].isna()
df["key"] = df["key"].fillna("Unknown")
df["in_shazam_charts_missing"] = missing_before["in_shazam_charts"]
decide("ausentes", "key vazio (tratamento)", df["key_missing"].sum(), 'categoria "Unknown" + key_missing', "nunca imputar C")
decide("ausentes", "in_shazam_charts vazio (tratamento)", df["in_shazam_charts_missing"].sum(),
       "NaN + indicador", "fora dos modelos principais")

print("Ausentes por coluna (CSV bruto):")
display(raw[EXPECTED_COLUMNS].isna().sum().loc[lambda s: s > 0].to_frame("n_ausentes"))

Ausentes por coluna (CSV bruto):


,n_ausentes
in_shazam_charts,50
key,95


## 4. Validações de faixa
Percentuais entre 0 e 100, `bpm > 0`, `artist_count >= 1`, contagens `>= 0` e data de lançamento válida. Cada falha vai para a tabela de anomalias.

In [18]:
for c in PCT_COLS:
    add_anomalies(df, (df[c] < 0) | (df[c] > 100), c, "percentual fora de 0-100")
add_anomalies(df, df["bpm"] <= 0, "bpm", "bpm <= 0")
add_anomalies(df, df["artist_count"] < 1, "artist_count", "artist_count < 1")
for c in COUNT_COLS:
    add_anomalies(df, df[c] < 0, c, "contagem negativa")

df["release_date"] = pd.to_datetime(
    dict(year=df["released_year"], month=df["released_month"], day=df["released_day"]), errors="coerce")
date_bad = df["release_date"].isna() & df[DATE_COLS].notna().all(axis=1)
add_anomalies(df, date_bad, "release_date", "data de lançamento inválida")
decide("validação", "data de lançamento inválida", date_bad.sum(), "registrado")

anom = pd.DataFrame(ANOMALIES)
print("Anomalias registradas até aqui:", len(anom))
if len(anom):
    display(anom.groupby(["column", "issue"]).size().to_frame("n"))

Anomalias registradas até aqui: 1


,,n
column,issue,
streams,valor não numérico (virou NaN),1


## 5. Duplicatas
Chave normalizada: `track_name` e `artist(s)_name` em minúsculas, sem acentos, sem pontuação e sem sufixos `(feat. …)`. Versões como `(Remix)` ou `(Taylor's Version)` **não** são removidas da chave, para não juntar músicas diferentes.

1. **Exatas** (todas as colunas iguais): removidas, mantendo a primeira.
2. **Não exatas:** classificadas em `quase_igual` (streams até 5% de diferença) ou `divergente`, e tratadas conforme `NON_EXACT_DUP_POLICY`. Com `"flag"`, só são marcadas. A decisão final fica em `docs/decisions.md`.

In [19]:
FEAT = r"[\(\[]\s*(?:feat|ft|featuring|with)\b[^\)\]]*[\)\]]"

def normalize_text(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c)).casefold()
    s = re.sub(FEAT, " ", s)
    s = re.sub(r"[\W_]+", " ", s).strip()
    return re.sub(r"\s(?:feat|ft|featuring)\s.*$", "", s)

def split_artists(v):
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return []
    return [a.strip() for a in str(v).split(",") if a.strip()]

df["track_key"] = df["track_name"].map(normalize_text)
df["artist_key"] = df["artist(s)_name"].map(lambda v: "|".join(sorted(normalize_text(a) for a in split_artists(v))))
df["song_group"] = df["track_key"] + "||" + df["artist_key"]   # grupo da CV por música (Etapa 3)

# 5.1 Exatas
exact = df.duplicated(subset=EXPECTED_COLUMNS, keep="first")
decide("duplicatas", "duplicatas exatas", exact.sum(), "removidas (mantida a primeira)")
removed_ids = list(df.loc[exact, "row_id"])
df = df[~exact].copy()

# 5.2 Não exatas
df["dup_group_size"] = df.groupby("song_group")["song_group"].transform("size")
df["dup_type"] = pd.NA
for _, g in df[df["dup_group_size"] > 1].groupby("song_group"):
    s = g["streams"]
    if s.isna().any():
        t = "divergente"
    else:
        rel = (s.max() - s.min()) / s.max() if s.max() else 0
        t = "quase_igual" if rel <= NEAR_DUP_STREAMS_RTOL else "divergente"
    df.loc[g.index, "dup_type"] = t

for t in ("quase_igual", "divergente"):
    n = int((df["dup_type"] == t).sum())
    if n:
        decide("duplicatas", f"linhas em grupos {t}", n, f"política: {NON_EXACT_DUP_POLICY}",
               "decisão final pendente" if NON_EXACT_DUP_POLICY == "flag" else "")

# 5.3 Política escolhida
if NON_EXACT_DUP_POLICY in ("keep_latest", "mean", "drop_all"):
    drop_idx = []
    for _, g in df[df["dup_type"].notna()].groupby("song_group"):
        if NON_EXACT_DUP_POLICY == "drop_all":
            drop_idx += list(g.index)
            continue
        keep = g.sort_values(["release_date", "row_id"]).index[-1]
        if NON_EXACT_DUP_POLICY == "mean":
            df.at[keep, "streams"] = g["streams"].mean()
        drop_idx += [i for i in g.index if i != keep]
    removed_ids += list(df.loc[drop_idx, "row_id"])
    df = df.drop(index=drop_idx)

print(f"Exatas removidas: {int(exact.sum())} | linhas removidas no total: {len(removed_ids)}")
dups = df[df["dup_type"].notna()].sort_values("song_group")
display(dups[["csv_line", "track_name", "artist(s)_name", "released_year", "streams", "dup_type"]])

Exatas removidas: 0 | linhas removidas no total: 0


,csv_line,track_name,artist(s)_name,released_year,streams,dup_type
372,374,About Damn Time,Lizzo,2022,723894473.0,quase_igual
764,766,About Damn Time,Lizzo,2022,723894473.0,quase_igual
178,180,SNAP,Rosa Linn,2022,726307468.0,quase_igual
873,875,SNAP,Rosa Linn,2022,711366595.0,quase_igual
223,225,Snow On The Beach (feat. More Lana Del Rey),"Lana Del Rey, Taylor Swift",2023,60350538.0,divergente
486,488,Snow On The Beach (feat. Lana Del Rey),"Taylor Swift, Lana Del Rey",2022,323437194.0,divergente
345,347,SPIT IN MY FACE!,ThxSoMch,2022,303216294.0,quase_igual
482,484,SPIT IN MY FACE!,ThxSoMch,2022,301869854.0,quase_igual
512,514,Take My Breath,The Weeknd,2021,130655803.0,divergente
616,618,Take My Breath,The Weeknd,2021,432702334.0,divergente


## 6. Colunas derivadas
As colunas originais não são alteradas. `age_years` usa `DATA_REFERENCE_DATE`.

In [20]:
df["age_years"] = (DATA_REFERENCE_DATE - df["release_date"]).dt.days / 365.25
df["release_period"] = pd.cut(df["released_year"], bins=PERIOD_BINS, labels=PERIOD_LABELS).astype("object")
df["log_streams"] = np.log1p(df["streams"])

lists = df["artist(s)_name"].map(split_artists)
df["artists_list"] = lists.map(lambda x: json.dumps(x, ensure_ascii=False))
df["main_artist"] = lists.map(lambda x: x[0] if x else pd.NA)
df["artist_count_mismatch"] = df["artist_count"] != lists.map(len)
add_anomalies(df, df["artist_count_mismatch"], "artist_count",
              "artist_count difere do nº de nomes em artist(s)_name")
decide("derivadas", "artist_count diverge do nº de nomes", df["artist_count_mismatch"].sum(),
       "marcado em artist_count_mismatch",
       "nomes com vírgula (ex.: 'Tyler, The Creator') também caem aqui")

print("Linhas com artist_count divergente:")
display(df.loc[df["artist_count_mismatch"], ["csv_line", "track_name", "artist(s)_name", "artist_count", "artists_list"]])
print(df["release_period"].value_counts(dropna=False).to_string())

Linhas com artist_count divergente:


,csv_line,track_name,artist(s)_name,artist_count,artists_list
667,669,"Cayï¿½ï¿½ La Noche (feat. Cruz Cafunï¿½ï¿½, Ab...","Quevedo, La Pantera, Juseph, Cruz Cafunï¿½ï¿½,...",7,"[""Quevedo"", ""La Pantera"", ""Juseph"", ""Cruz Cafu..."
742,744,Vampiro,"Matuï¿½ï¿½, Wiu,",3,"[""Matuï¿½ï¿½"", ""Wiu""]"


release_period
2022         402
2018-2021    202
2023         175
2000-2017    125
pre-2000      49


## 7. Elegibilidade para a modelagem supervisionada
A linha com `streams` inválido **não é apagada**: recebe `usable_for_supervised=False` e continua nas análises que não dependem do alvo.

In [21]:
df["usable_for_supervised"] = df["streams"].notna() & (df["streams"] >= 0)
df["exclusion_reason"] = np.where(df["usable_for_supervised"], pd.NA, "streams ausente/não numérico")
decide("alvo", "linhas sem streams válido", (~df["usable_for_supervised"]).sum(),
       "usable_for_supervised=False", "fora da modelagem supervisionada; mantidas na EDA sem alvo")
display(df.loc[~df["usable_for_supervised"], ["csv_line", "track_name", "streams", "exclusion_reason"]])

,csv_line,track_name,streams,exclusion_reason
574,576,Love Grows (Where My Rosemary Goes),NaN,streams ausente/não numérico


## 8. Onde se concentram as chaves ausentes
Verifica se `key` vazia se concentra em algum ano ou artista (para saber se a ausência é aleatória).

In [25]:
key_by_year = (df.groupby("released_year")["key_missing"].agg(n="size", n_missing="sum")
               .assign(pct=lambda t: (100 * t.n_missing / t.n).round(1)).reset_index())
key_by_artist = (df.groupby("main_artist")["key_missing"].agg(n="size", n_missing="sum")
                 .sort_values("n_missing", ascending=False).head(15).reset_index())
print("Por ano (anos com ausência):")
display(key_by_year[key_by_year["n_missing"] > 0])
print("Top 15 artistas por nº de chaves ausentes:")
display(key_by_artist)

Por ano (anos com ausência):


,released_year,n,n_missing,pct
4,1952,1,1,100.0
9,1968,1,1,100.0
13,1975,2,1,50.0
17,1984,4,1,25.0
29,2000,4,1,25.0
32,2004,4,1,25.0
34,2007,1,1,100.0
36,2010,7,1,14.3
37,2011,10,1,10.0
38,2012,10,1,10.0


Top 15 artistas por nº de chaves ausentes:


,main_artist,n,n_missing
0,The Weeknd,34,7
1,Taylor Swift,36,5
2,Harry Styles,17,4
3,Billie Eilish,6,3
4,Peso Pluma,5,3
5,Bad Bunny,26,3
6,Arctic Monkeys,4,2
7,Post Malone,6,2
8,Karol G,9,2
9,SZA,23,2


## 9. Verificações finais
Substituem os testes automáticos: se qualquer `assert` falhar, o notebook para.

In [23]:
# Ordem das colunas: originais primeiro, derivadas depois
front = ["row_id", "csv_line"] + EXPECTED_COLUMNS
df = df[front + [c for c in df.columns if c not in front]].reset_index(drop=True)

assert pd.api.types.is_numeric_dtype(df["streams"]), "streams não numérico"
for c in COUNT_COLS:
    assert pd.api.types.is_numeric_dtype(df[c]), f"{c} não numérico"
assert not df["key"].isna().any(), "key com ausentes"
assert len(df) == N_RAW - len(removed_ids), "contagem de linhas não bate com o log"
assert df.loc[df["usable_for_supervised"], "streams"].notna().all()
assert not df["row_id"].duplicated().any()
print(f"OK: {N_RAW} linhas brutas - {len(removed_ids)} removidas = {len(df)} linhas "
      f"({int(df['usable_for_supervised'].sum())} usáveis no supervisionado)")

OK: 953 linhas brutas - 0 removidas = 953 linhas (952 usáveis no supervisionado)


## 10. Salvar saídas e relatório de qualidade

In [24]:
df.to_csv(PROCESSED_DIR / "tracks_clean.csv", index=False, encoding="utf-8")
try:
    df.to_parquet(PROCESSED_DIR / "tracks_clean.parquet", index=False)
except ImportError:
    print("pyarrow ausente: gerado só o CSV limpo.")

log_df, anom_df = pd.DataFrame(LOG), pd.DataFrame(ANOMALIES)
log_df.to_csv(TABLES_DIR / "cleaning_log.csv", index=False)
anom_df.to_csv(TABLES_DIR / "anomalies.csv", index=False)
key_by_year.to_csv(TABLES_DIR / "key_missing_by_year.csv", index=False)
key_by_artist.to_csv(TABLES_DIR / "key_missing_by_artist.csv", index=False)

lines = ["# Relatório de qualidade dos dados", "",
         f"- Linhas no CSV bruto: **{N_RAW}**",
         f"- Linhas removidas: **{len(removed_ids)}**",
         f"- Linhas no arquivo limpo: **{len(df)}**",
         f"- Linhas usáveis na modelagem supervisionada: **{int(df['usable_for_supervised'].sum())}**",
         f"- Artistas principais distintos: **{df['main_artist'].nunique()}**", "",
         "| Etapa | Item | n | Ação | Motivo |", "| --- | --- | ---: | --- | --- |"]
lines += [f"| {r.etapa} | {r.item} | {r.n} | {r.acao} | {r.motivo} |" for r in log_df.itertuples()]
(REPORTS_DIR / "quality_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

# O CSV original não pode ter mudado
(PROCESSED_DIR / "raw_sha256.txt").write_text(HASH_BEFORE + "\n")
assert sha256_file(RAW_CSV) == HASH_BEFORE, "O CSV bruto foi modificado!"
print("CSV bruto intacto. Saídas em", PROCESSED_DIR, "e", REPORTS_DIR)
display(log_df)

CSV bruto intacto. Saídas em /home/zana/projects/facul/intML/ecp-records-project/dataset/processed e /home/zana/projects/facul/intML/ecp-records-project/reports


,etapa,item,n,acao,motivo
0,leitura,nomes com possível mojibake,0,mantidos e registrados,lido em cp1252; conferir manualmente se houver...
1,numérico,valor não numérico em streams,1,convertido para NaN e registrado,
2,numérico,separador de milhares em in_deezer_playlists,79,vírgula removida,
3,numérico,separador de milhares em in_shazam_charts,7,vírgula removida,
4,ausentes,in_shazam_charts vazio,50,tratado abaixo,
5,ausentes,key vazio,95,tratado abaixo,
6,ausentes,key vazio (tratamento),95,"categoria ""Unknown"" + key_missing",nunca imputar C
7,ausentes,in_shazam_charts vazio (tratamento),50,NaN + indicador,fora dos modelos principais
8,validação,data de lançamento inválida,0,registrado,
9,duplicatas,duplicatas exatas,0,removidas (mantida a primeira),


## 11. Pendências para fechar em `docs/decisions.md`
- Duplicatas não exatas: manter o mais recente, tirar a média ou excluir todos? (hoje: apenas marcadas)
- Período de lançamento no modelo: `released_year`, `release_period` ou `age_years`?
- Confirmar no Kaggle: data de coleta, significado de `streams` (acumulado?) e codificação das colunas de charts.
- Conferir a linha com `artist_count` divergente e a linha de `streams` corrompido.